# 03 — Benchmarks, comparisons and deck figures

The leakage-safe reveal loop: at each step reveal one position, refit everything from the revealed block, reconstruct all 101 positions, score on the withheld ones.

Drivers (each ~5–15 min): `run_phys.py` (EB/EB+GP/low-rank), `run_fem.py <lib> <reps> <tag> <csv>` for any library (FEM, EB variants), `run_gp.py` (GP-only + oracle-ℓ diagnostic). This notebook reads the CSVs they produce.

In [ ]:
import sys; sys.path.insert(0,'..'); sys.path.insert(0,'../src')
import pandas as pd
from config import OUT
D=pd.concat([pd.read_csv(OUT/f) for f in ('phys_bench.csv','fem_bench.csv','gp_bench.csv')],ignore_index=True)
E=D[D.strategy=='equispaced']
E.pivot_table(index='n',columns='rec',values='amp_map_pct').round(2)

### Positions needed (equispaced)

| target | low-rank | GP only | EB | FEM | EB+GP | FEM+GP |
|---|---|---|---|---|---|---|
| NRMSE ≤ 10 % | 12 | 5 | 5 | **4** | 5 | **4** |
| NRMSE ≤ 5 % | 20 | 7 | 30 | never | 5 | 5 |
| NRMSE ≤ 2 % | never | 20 | never | never | 12 | **8** |
| D-NS ≤ 0.5 µm | 30 | 12 | 30 | **4** | 16 | **4** |

Caveats that must travel with these numbers:
1. bare FEM's 0.48 µm D-NS is a **model bias**, constant from n=4 to 16 (0.04 µm spread over 24 random designs) — quote FEM+GP for convergence claims;
2. the FEM arm is **isotropic** while EB has k₂=0 — geometry and lateral spring are conflated until the frictionless library / EB-iso-cone arm settle it;
3. acquisition strategy is not the lever — equispaced ≈ D-optimal ≫ nothing; the model choice moves error ~10× further.

In [ ]:
d=E[E.rec.isin(['gp','fem_gp'])].pivot_table(index='n',columns='rec',values='amp_map_pct')
(d['gp']/d['fem_gp']).round(2).rename('physics premium (GP-only / FEM+GP)')

### Figures

Each deck figure is one script in `figures/` (ORNL template palette, colour = forward model, line style = ±GP):

- `d_models.py` — the two forward models, HAS/LACKS, drawn from the actual mesh
- `d_figs.py` — ground truth, transfer functions, mode shapes, 2-D recon, Q6, Q7, caveat
- `d_al.py` / `d_acq.py` / `d_gp.py` / `d_modes.py` — AL panel, acquisition, GP baseline, mode-ratio identifiability

`reporting/build_deck.py` assembles the 21-slide deck (needs `ORNL template 1.pptm` beside it; note the macroEnabled content-type fix inside — a .pptx built from a .pptm template will not open in PowerPoint without it).

In [ ]:
import subprocess, glob
# regenerate everything (uncomment):
# for s in sorted(glob.glob('../figures/d_*.py')): subprocess.run([sys.executable,s],cwd='../figures',check=True)
sorted(glob.glob(str(OUT/'fig'/'*.png')))